# 第7回 特徴量エンジニアリング

本ノートでは、スライドで紹介した特徴量エンジニアリングの手法を、titanic データセットを使って実際に動かしながら学んでいきましょう。

**ゴール**: 特徴量エンジニアリングの代表的な手法を、データに応じて適切に使い分けられるようになる。


## 目次

1. 特徴量エンジニアリングの基本
2. 欠損値の扱い
3. 数値変数の変換
4. カテゴリカル変数の変換
5. 集約特徴量と仮説ベースの特徴量作成
6. データリーケージと実践時の注意点
7. 特徴量の選択と評価


## ライブラリの読み込み

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# 日本語フォントの設定(import するだけで matplotlib に日本語フォントが適用される)
try:
    import japanize_matplotlib  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'])
    import japanize_matplotlib  # noqa: F401

# 乱数の固定
RANDOM_STATE = 42


# 1. 特徴量エンジニアリングの基本

本章では、seaborn の **titanic データセット**、つまりタイタニック号の乗客生存予測を題材に、特徴量エンジニアリングの手法を実践していきます。まずは titanic データを読み込み、中身を確認してみましょう。


In [ ]:
df = sns.load_dataset('titanic')
print('shape:', df.shape)
df.head()


## titanic データセットの主な列

扱う各列の意味を整理しておきます。本講義では `survived` を目的変数として、それ以外の列から特徴量を作っていきます。

| 列名 | 説明 |
|---|---|
| `survived` | 生存。0=死亡、1=生存。**今回の目的変数** |
| `pclass` | チケット等級。1=1等、2=2等、3=3等 |
| `sex` | 性別。male、female |
| `age` | 年齢。欠損あり |
| `sibsp` | 同乗していた兄弟姉妹・配偶者の数 |
| `parch` | 同乗していた親・子供の数 |
| `fare` | 運賃 |
| `embarked` | 乗船港。C、Q、S の3種類 |
| `class` | `pclass` の文字列版。First、Second、Third |
| `who` | 属性。man、woman、child |
| `adult_male` | 成人男性かどうかの真偽値 |
| `deck` | 客室のデッキ。A〜G、欠損が非常に多い |
| `embark_town` | 乗船港の名前 |
| `alive` | 生存の文字列版。yes、no |
| `alone` | 1人で乗船したかの真偽値 |

なお `class`、`who`、`adult_male`、`alive`、`alone` などは、`pclass` や `sex`、`age` から派生してできた列です。


欠損値の状況も確認しておきましょう。


In [ ]:
df.isna().sum().sort_values(ascending=False)


後の章で特徴量を追加していくたびに改善幅を測れるよう、まずは最小限の特徴量だけでベースラインモデルを学習してみましょう。以降の章ではこのCVスコアを基準にします。


まずは特徴量を組み立てて、テーブルの状態を確認します。最小限の特徴量として、数値で欠損のない `pclass`, `sibsp`, `parch`, `fare` の4つを使います。`fare` には1件だけ欠損があるので、ここでは中央値で埋めておきます。詳しい欠損補完は2章で扱います。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

baseline_cols = ['pclass', 'sibsp', 'parch', 'fare']
X_baseline = df[baseline_cols].copy()
X_baseline['fare'] = X_baseline['fare'].fillna(X_baseline['fare'].median())
y = df['survived']

print(f'X_baseline の特徴量 {X_baseline.shape[1]}個: {X_baseline.columns.tolist()}')
print(f'サンプル数: {X_baseline.shape[0]}')
X_baseline.head()


特徴量を確認できたので、ベースラインの CV スコアを評価します。以降の章ではこのスコアを基準に、特徴量を加えた効果を測っていきます。


In [ ]:
models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
    ]),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
}

baseline_scores = {}
for name, model in models.items():
    score = cross_val_score(model, X_baseline, y, cv=5, scoring='accuracy').mean()
    baseline_scores[name] = score
    print(f'{name}: CV accuracy = {score:.4f}')


#### 練習問題 1

titanic データセットで、チケット等級 `pclass` と生存 `survived` の関係を、表または棒グラフで確認してみましょう。

**ヒント**: `df.groupby('pclass')['survived'].mean()` でクラスごとの生存率を集計できます。


In [ ]:
# 練習問題1の答えはこのセルに書いてください


# 2. 欠損値の扱い

本章では、titanic の `age`、`embarked`、`deck` の欠損に、中央値・最頻値補完と欠損フラグを実際に試してみましょう。


まず `age` で平均値と中央値の位置を見比べてみます。どちらで補完するかで埋まる値が変わります。


In [ ]:
print('age 平均値:', df['age'].mean())
print('age 中央値:', df['age'].median())

fig, ax = plt.subplots(figsize=(8, 4))
df['age'].hist(bins=30, ax=ax, color='steelblue', alpha=0.7)
ax.axvline(df['age'].mean(), color='orange', linestyle='--', label=f"平均 {df['age'].mean():.1f}")
ax.axvline(df['age'].median(), color='red', linestyle='--', label=f"中央値 {df['age'].median():.1f}")
ax.set_xlabel('age')
ax.set_ylabel('count')
ax.legend()
plt.show()


ヒストグラムは**右に少し裾を引いた形**になっており、それに引っ張られて**平均が中央値より右側にズレている**のが見て取れます。こうした分布では平均が一部の高い値の影響を受けやすいので、**中央値で補完するほうが安全**です。


外れ値に強い中央値で `age` を補完してみましょう。`fillna()` を使うとシンプルに書けます。


In [ ]:
df_imp = df.copy()

age_median = df_imp['age'].median()
print('age の補完前の欠損数:', df_imp['age'].isna().sum())
df_imp['age'] = df_imp['age'].fillna(age_median)
print('age の補完後の欠損数:', df_imp['age'].isna().sum())
print('補完に使った値:', age_median)


`embarked` は文字列のカテゴリ変数なので最頻値で、`fare` も1件欠損があるので中央値で補完してみましょう。


In [ ]:
embarked_mode = df_imp['embarked'].mode()[0]
print('embarked の最頻値:', embarked_mode)
df_imp['embarked'] = df_imp['embarked'].fillna(embarked_mode)
print('embarked の欠損数:', df_imp['embarked'].isna().sum())

df_imp['fare'] = df_imp['fare'].fillna(df_imp['fare'].median())
print('fare の欠損数:', df_imp['fare'].isna().sum())


`deck` は688件も欠損しているので、補完するより「欠損していること自体」を `deck_missing` という特徴量にしてみましょう。


In [ ]:
df_imp['deck_missing'] = df_imp['deck'].isna().astype(int)
df_imp[['deck', 'deck_missing']].head(10)


欠損フラグそのものに予測力があるかも見ておきましょう。`deck` が欠損している人と欠損していない人で生存率に差があれば、特徴量として効きそうです。


In [ ]:
df_imp.groupby('deck_missing')['survived'].mean()


`deck` が記録されている人は生存率が約0.67、欠損している人は約0.30と、**明確な差**が出ています。つまり「欠損しているという事実」自体が予測に役立つ情報を持っているということです。欠損を埋めるだけでなく、**欠損フラグを特徴量として残す価値がある**と判断できます。


ここまでの欠損処理を踏まえて、特徴量を整理してから、ベースラインからCVスコアがどう変わったか確認してみましょう。


In [ ]:
ch2_cols = baseline_cols + ['deck_missing']
X_ch2 = df_imp[ch2_cols].copy()

print(f'本章で追加: deck_missing')
print(f'X_ch2 の特徴量 {X_ch2.shape[1]}個: {X_ch2.columns.tolist()}')
X_ch2.head()


特徴量の状態が確認できたので、CV スコアの変化を見てみます。Δ はベースラインからの差分です。


In [ ]:
for name, model in models.items():
    score = cross_val_score(model, X_ch2, y, cv=5, scoring='accuracy').mean()
    diff = score - baseline_scores[name]
    print(f'{name}: CV accuracy = {score:.4f}  (Δ {diff:+.4f})')


#### 練習問題 2

`embarked` を最頻値で補完する代わりに、`'Unknown'` という新しいカテゴリで補完してみましょう。補完後の `embarked` について、各カテゴリの件数を確認してください。

**ヒント**: `df_imp['embarked'].fillna('Unknown')` で補完できます。


In [ ]:
# 練習問題2の答えはこのセルに書いてください


# 3. 数値変数の変換

本章では `age` や `fare` のような数値変数を、モデルに伝わりやすい形に変換してみましょう。スケーリング・対数変換・クリッピング・比率・交差項を順に試します。


まず運賃 `fare` の分布をヒストグラムと箱ひげ図で確認してみます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_imp['fare'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('fare のヒストグラム')
axes[0].set_xlabel('fare')

axes[1].boxplot(df_imp['fare'].dropna(), vert=False)
axes[1].set_title('fare の箱ひげ図')
axes[1].set_xlabel('fare')
plt.tight_layout()
plt.show()

df_imp['fare'].describe()


**ヒストグラムは大きく右に裾を引いた形**で、ほとんどの乗客は0〜50程度の安い運賃に集中し、一部だけ200を超える高額運賃を払っています。箱ひげ図でも上側に外れ値が多く見えます。このまま線形モデルに入れると少数の高額運賃に予測が引っ張られる可能性があるため、**対数変換やクリッピングが効きそう**な分布だと判断できます。


次にスケーリングを試してみましょう。`age` と `fare` を StandardScaler で標準化し、平均と標準偏差がどう変わるか確認します。


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(df_imp[['age', 'fare']])
df_scaled = pd.DataFrame(scaled, columns=['age_std', 'fare_std'])

print('標準化前:')
print(df_imp[['age', 'fare']].describe().loc[['mean', 'std']])
print()
print('標準化後:')
print(df_scaled.describe().loc[['mean', 'std']])


標準化前は `age` の標準偏差が約13、`fare` は約50と、ばらつきの大きさが4倍近く違っていました。標準化後は **両方とも平均0・標準偏差1にそろい、同じレンジで比較できる状態** になります。線形回帰やkNNなど距離・大きさを使うモデルでは、この処理がモデル精度に直結します。


続いて `fare` を対数変換してみましょう。右に長く伸びた分布が左右対称に近づくはずです。`np.log1p` を使うと `log(1 + fare)` を計算できます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_imp['fare'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('fare(対数変換前)')
axes[0].set_xlabel('fare')

fare_log = np.log1p(df_imp['fare'])
fare_log.hist(bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('fare(対数変換後)')
axes[1].set_xlabel('log(1 + fare)')
plt.tight_layout()
plt.show()


このように、対数変換によって **右に長く伸びていた分布が左右対称に近い形** になりました。少数の高額運賃の影響が抑えられ、線形モデルが学習しやすい形になっています。


対数変換した値を `fare_log` として保存しておきます。


In [ ]:
df_imp['fare_log'] = np.log1p(df_imp['fare'])


外れ値の影響を抑えるためのクリッピングも試してみましょう。`fare` の99パーセンタイル値を上限にして丸めます。


In [ ]:
upper = df_imp['fare'].quantile(0.99)
print(f'99パーセンタイル(上限): {upper:.1f}')

df_imp['fare_clipped'] = df_imp['fare'].clip(upper=upper)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_imp['fare'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('fare(クリッピング前)')
axes[0].axvline(upper, color='red', linestyle='--', label=f'上限 {upper:.1f}')
axes[0].legend()

df_imp['fare_clipped'].hist(bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('fare(クリッピング後)')
plt.tight_layout()
plt.show()


クリッピング後は **上限を超えていた極端な値がすべて上限値に集約** され、右端に1本の棒として現れています。対数変換とは違い、**中央の山の形はあまり変えず、極端な外れ値だけを抑える**のがクリッピングの特徴です。


比率特徴量も作ってみましょう。家族の人数 `sibsp + parch + 1` で運賃を割れば、1人あたりの運賃を表せます。


In [ ]:
df_imp['family_size'] = df_imp['sibsp'] + df_imp['parch'] + 1  # 本人含む
df_imp['fare_per_person'] = df_imp['fare'] / df_imp['family_size']
df_imp[['fare', 'family_size', 'fare_per_person']].head()


最後に交差項です。`age × pclass` を作って、「若くて1等の人」のような組み合わせ特徴を伝えてみましょう。


In [ ]:
df_imp['age_x_pclass'] = df_imp['age'] * df_imp['pclass']
df_imp[['age', 'pclass', 'age_x_pclass']].head()


本章で作った特徴量を加え、整理してからCVスコアを確認してみましょう。


In [ ]:
ch3_cols = ['pclass', 'sibsp', 'parch', 'fare_log', 'fare_per_person', 'family_size', 'age_x_pclass', 'deck_missing']
X_ch3 = df_imp[ch3_cols].copy()

print('本章で追加: fare_log, fare_per_person, family_size, age_x_pclass')
print('代わりに外したもの: fare、対数変換した fare_log で置き換え')
print(f'X_ch3 の特徴量 {X_ch3.shape[1]}個: {X_ch3.columns.tolist()}')
X_ch3.head()


特徴量の状態が確認できたので、CV スコアの変化を見てみます。


In [ ]:
for name, model in models.items():
    score = cross_val_score(model, X_ch3, y, cv=5, scoring='accuracy').mean()
    diff = score - baseline_scores[name]
    print(f'{name}: CV accuracy = {score:.4f}  (Δ {diff:+.4f})')


#### 練習問題 3

`age` を `np.log1p(df_imp['age'])` で対数変換した特徴量を作り、ヒストグラムで対数変換前後の分布を比較してみましょう。

**ヒント**: 既存の対数変換のコードを参考に。


In [ ]:
# 練習問題3の答えはこのセルに書いてください


# 4. カテゴリカル変数の変換

本章では `sex`、`embarked`、`class` のようなカテゴリカル変数を、いくつかの方法で数値化してみましょう。Label Encoding、One-Hot Encoding、Frequency Encoding、Target Encoding を順に試します。


まず `embarked` に Label Encoding を試してみましょう。sklearn の `LabelEncoder` でカテゴリごとに整数を割り当てます。


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_imp['embarked_label'] = le.fit_transform(df_imp['embarked'])

# 対応関係を確認
for cat, num in zip(le.classes_, range(len(le.classes_))):
    print(f'{cat} → {num}')

df_imp[['embarked', 'embarked_label']].head()


このように `C → 0, Q → 1, S → 2` のように整数が割り当てられました。**ただし「0, 1, 2」の数字の大小に本来は意味がありません**。たとえば「C は S の半分」のような関係は存在しないのに、線形モデルではこの大小をそのまま受け取って学習してしまうため、不自然な関係が混入する可能性があります。


実際に、線形モデルが Label Encoding を「順序のある数値」として扱ってしまう様子を見てみましょう。本来は順序のない `embarked` を Label Encoding 1列だけで生存予測してみます。


In [ ]:
from sklearn.linear_model import LogisticRegression

# C=0, Q=1, S=2 を1列だけで学習する
X_label = df_imp[['embarked_label']]
lr = LogisticRegression(random_state=RANDOM_STATE)
lr.fit(X_label, y)

# 各カテゴリでの予測生存率
for cat, num in zip(le.classes_, range(len(le.classes_))):
    proba = lr.predict_proba([[num]])[0, 1]
    print(f'{cat} (label={num}) の予測生存率: {proba:.3f}')


label が 0→1→2 と増えるにつれて、予測生存率が 0.54→0.44→0.34 と **単調に下がっている** のが分かります。線形モデルは Label Encoding の値を「順序のある量」として扱うため、**label の並び順に沿った単調な関係しか学習できません**。今回はたまたま `C, Q, S` の実際の生存率もこの順序に並んでいたので大きく外していませんが、もし `LabelEncoder` のアルファベット順と本来の順序が一致していなかったら、**カテゴリ間の関係をうまく表現できなくなります**。本来順序を持たないカテゴリ変数に対しては、One-Hot Encoding のほうが安全だと考えられます。


次に One-Hot Encoding を試してみましょう。`pd.get_dummies` を使うと、カテゴリごとに 0/1 の列を作れます。


In [ ]:
embarked_ohe = pd.get_dummies(df_imp['embarked'], prefix='embarked').astype(int)
embarked_ohe.head()


このように、各カテゴリが独立した 0/1 の列になりました。カテゴリ間に大小関係は持ち込まれないため、**線形モデルでも安全に使えます**。一方で、カテゴリ数が多いと列がそのぶん増える点には注意が必要です。


Count Encoding と Frequency Encoding も試してみましょう。Count Encoding は各カテゴリを出現回数で、Frequency Encoding は出現割合で置き換えます。


In [ ]:
# Count Encoding: 出現回数で置換
count = df_imp['embarked'].value_counts()
df_imp['embarked_count'] = df_imp['embarked'].map(count)

# Frequency Encoding: 出現割合で置換
freq = df_imp['embarked'].value_counts(normalize=True)
df_imp['embarked_freq'] = df_imp['embarked'].map(freq)

print('Count Encoding と Frequency Encoding の対応:')
print(pd.DataFrame({'count': count, 'freq': freq.round(3)}))
print()
df_imp[['embarked', 'embarked_count', 'embarked_freq']].head()


出現が多いカテゴリほど大きい数値になります。**出現頻度そのものに意味がある**ような場面で効果的です。たとえば「よく買われる商品ほど在庫が多い」「人気店舗ほど客数が多い」のように、頻度が他の指標と連動するときに使えます。One-Hot より列数を抑えられるのも利点です。


Target Encoding では、カテゴリごとの目的変数の平均で置き換えます。`embarked` ごとの生存率で `embarked` を数値化してみましょう。なお、この手法はリーケージの危険があるため、安全な運用は6章で詳しく扱います。


In [ ]:
target_mean = df_imp.groupby('embarked')['survived'].mean()
print('embarked ごとの生存率:')
print(target_mean.round(3))

df_imp['embarked_target'] = df_imp['embarked'].map(target_mean)
df_imp[['embarked', 'embarked_target']].head()


C 0.55, Q 0.39, S 0.34 のように、**カテゴリの「予測したい変数との関係の強さ」が1列に圧縮されます**。少ない列数で強い特徴量になりやすい一方、目的変数を直接使うため**リーケージの危険が常につきまといます**。安全に使うには「学習データだけから計算した値を、検証・本番データに transform する」という運用が必須です。詳しくは6章で扱います。


推論時に未知のカテゴリが現れる場合への備えも見ておきましょう。sklearn の `OneHotEncoder` で `handle_unknown='ignore'` を指定すると、未知カテゴリの行はすべての列が0になり、エラーなく処理できます。


In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_imp[['embarked']])

# 学習時にないカテゴリを推論しても落ちない
unknown = pd.DataFrame({'embarked': ['S', 'C', 'Q', 'X']})  # X は未知
result = ohe.transform(unknown)
pd.DataFrame(result, columns=ohe.get_feature_names_out(['embarked']))


`sex`、`embarked`、`class` を One-Hot Encoding して特徴量に加え、整理してからCVスコアを確認してみましょう。


In [ ]:
sex_ohe = pd.get_dummies(df_imp['sex'], prefix='sex').astype(int)
embarked_ohe = pd.get_dummies(df_imp['embarked'], prefix='embarked').astype(int)
class_ohe = pd.get_dummies(df_imp['class'], prefix='class').astype(int)

X_ch4 = pd.concat([
    df_imp[['pclass', 'sibsp', 'parch', 'fare_log', 'fare_per_person', 'family_size', 'age_x_pclass', 'deck_missing']],
    sex_ohe, embarked_ohe, class_ohe,
], axis=1)

print('本章で追加: sex を One-Hot で2列、embarked を3列、class を3列')
print(f'X_ch4 の特徴量 {X_ch4.shape[1]}個: {X_ch4.columns.tolist()}')
X_ch4.head()


特徴量の状態が確認できたので、CV スコアの変化を見てみます。


In [ ]:
for name, model in models.items():
    score = cross_val_score(model, X_ch4, y, cv=5, scoring='accuracy').mean()
    diff = score - baseline_scores[name]
    print(f'{name}: CV accuracy = {score:.4f}  (Δ {diff:+.4f})')


#### 練習問題 4

`who` カラムは man、woman、child の3値を持ちます。これに One-Hot Encoding を適用し、`X_ch4` にさらに追加してCVスコアを確認してみましょう。`sex` と情報が重なる部分もありますが、「子供かどうか」の情報が新たに加わるはずです。


In [ ]:
# 練習問題4の答えはこのセルに書いてください


# 5. 集約特徴量と仮説ベースの特徴量作成

本章では titanic に戻り、`pclass` や `embarked` を単位とした集約特徴量と、業務知識から発想する仮説ベースの特徴量を作ってみましょう。


まず `pclass` ごとの平均運賃を集約特徴量として作ってみましょう。pandas の `groupby` を使えば簡単です。


In [ ]:
pclass_mean_fare = df_imp.groupby('pclass')['fare'].mean()
print('pclass ごとの平均運賃:')
print(pclass_mean_fare)

df_imp['pclass_mean_fare'] = df_imp['pclass'].map(pclass_mean_fare)
df_imp[['pclass', 'fare', 'pclass_mean_fare']].head()


集約は平均だけでなく件数や合計などいろいろ計算できます。`embarked` ごとに、生存率を mean、乗客数を count で一気に出してみましょう。


In [ ]:
df_imp.groupby('embarked')['survived'].agg(['mean', 'count'])


集約値と個別の値を比較する特徴量も作ってみましょう。「自分が払った運賃 ÷ 同じ等級の平均運賃」を計算すると、等級内で平均より高いチケットを買ったかが分かります。


In [ ]:
df_imp['fare_vs_pclass_mean'] = df_imp['fare'] / df_imp['pclass_mean_fare']
df_imp[['fare', 'pclass_mean_fare', 'fare_vs_pclass_mean']].head()


次に仮説ベースの特徴量です。titanic では「女性と子供が優先された」が有名な仮説です。これに沿って、`is_child`、`is_alone`、`is_woman_or_child` を作り、それぞれグループの生存率を比較してみましょう。


In [ ]:
df_imp['is_child'] = (df_imp['age'] < 13).astype(int)
df_imp['is_alone'] = (df_imp['family_size'] == 1).astype(int)
df_imp['is_woman_or_child'] = ((df_imp['sex'] == 'female') | (df_imp['age'] < 13)).astype(int)

print('is_child の生存率:')
print(df_imp.groupby('is_child')['survived'].mean())
print()
print('is_woman_or_child の生存率:')
print(df_imp.groupby('is_woman_or_child')['survived'].mean())


女性または子供のグループ、つまり `is_woman_or_child = 1` の人々は、**それ以外のグループより圧倒的に生存率が高い** ことが数値で確認できました。これは「女性と子供が優先された」という仮説と整合しており、`is_woman_or_child` は予測に効きそうな特徴量だと判断できます。


本章で作った特徴量を加え、整理してからCVスコアを確認してみましょう。


In [ ]:
X_ch5 = pd.concat([X_ch4, df_imp[['pclass_mean_fare', 'fare_vs_pclass_mean', 'is_child', 'is_alone', 'is_woman_or_child']]], axis=1)

print('本章で追加: pclass_mean_fare, fare_vs_pclass_mean, is_child, is_alone, is_woman_or_child')
print(f'X_ch5 の特徴量 {X_ch5.shape[1]}個: {X_ch5.columns.tolist()}')
X_ch5.head()


特徴量の状態が確認できたので、CV スコアの変化を見てみます。


In [ ]:
for name, model in models.items():
    score = cross_val_score(model, X_ch5, y, cv=5, scoring='accuracy').mean()
    diff = score - baseline_scores[name]
    print(f'{name}: CV accuracy = {score:.4f}  (Δ {diff:+.4f})')


#### 練習問題 5

`sex` を単位として `fare` の平均をとった集約特徴量を作り、各乗客にその値を割り当ててみましょう。

**ヒント**: `df_imp.groupby('sex')['fare'].mean()` で集約値が出ます。


In [ ]:
# 練習問題5の答えはこのセルに書いてください


# 6. データリーケージと実践時の注意点

本章では Target Encoding を題材に、データリーケージがどう手元のスコアを偽装してしまうかを実験で見てみましょう。「もし完全にランダムな乗客IDを Target Encoding したらどうなるか?」を考えます。本来予測力はゼロのはずです。


まず、titanic データに 0〜199 の整数を完全にランダムに割り振った `random_id` 列を作って加えます。これは目的変数とは何の関係もない乱数なので、本来予測には全く役立たない「ダミー特徴量」です。


In [ ]:
df_lk = df_imp.copy()
rng = np.random.RandomState(RANDOM_STATE)
df_lk['random_id'] = rng.randint(0, 200, size=len(df_lk))

print('random_id のカテゴリ数:', df_lk['random_id'].nunique())
print(f'1カテゴリあたりの平均サンプル数: {len(df_lk) / df_lk["random_id"].nunique():.1f}')


NG例として、全データから `random_id` の Target Encoding、つまりカテゴリごとの生存率を計算してから、その特徴量で CV を回してみましょう。


In [ ]:
from sklearn.model_selection import KFold

# NG: 全データで Target Encoding を計算してから CV にかける
target_mean = df_lk.groupby('random_id')['survived'].mean()
df_lk['random_id_te'] = df_lk['random_id'].map(target_mean)

base_cols = ['pclass', 'age', 'fare', 'sibsp', 'parch']
X_leak = df_lk[base_cols + ['random_id_te']]

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
])
score_leak = cross_val_score(pipe, X_leak, y, cv=5, scoring='accuracy').mean()
print(f'NG (リーケージあり): CV accuracy = {score_leak:.4f}')


CVスコアが0.75前後の高い値になりました。乱数なのに「効いている」ように見えています。次に、リーケージを防ぐ正しい方法で同じことをやってみましょう。fold ごとに学習データだけから Target Encoding を fit し、検証データには同じ値を transform するだけにします。


In [ ]:
# OK: fold ごとに train だけで Target Encoding を計算し、valid には同じ値を transform
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = []
for tr, va in kf.split(df_lk):
    tm = df_lk.iloc[tr].groupby('random_id')['survived'].mean()
    overall = df_lk.iloc[tr]['survived'].mean()

    X_tr = df_lk.iloc[tr][base_cols].copy()
    X_va = df_lk.iloc[va][base_cols].copy()
    X_tr['random_id_te'] = df_lk.iloc[tr]['random_id'].map(tm).fillna(overall).values
    X_va['random_id_te'] = df_lk.iloc[va]['random_id'].map(tm).fillna(overall).values

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
    ])
    model.fit(X_tr, y.iloc[tr])
    scores.append(model.score(X_va, y.iloc[va]))

score_ok = sum(scores) / len(scores)
print(f'OK (リーケージなし): CV accuracy = {score_ok:.4f}')
print(f'差分: {score_leak - score_ok:+.4f}')


CVスコアが大きく下がり、ほぼ「ベースラインのみで予測した場合」と同じ水準になりました。これが `random_id` の本当の予測力で、要するにほぼゼロです。つまり NG 版は「効いているように見えていただけ」で、本番に出したら全く再現しないスコアでした。

なぜこうなるのかというと、Target Encoding の値を全データから計算すると、各行の値には「その行自身の `survived`」が混ざってしまうからです。特にカテゴリ内のサンプル数が少ないと、自分1人の影響が大きく、ほぼ自分の答えそのものになります。


## fit / transform で同じルールを使い回す

上のNG/OK実験で分かったのは、**「補完・エンコーディング・スケーリングの計算ルールは学習データだけから決め、検証・本番には同じ値をそのまま当てはめる」必要がある**ということです。これを sklearn では `fit` と `transform` の2つに分けて表現します。`fit` は学習データからルールを決める操作、`transform` は決めたルールでデータを変換する操作です。

**ルール**: 学習データには `fit` または `fit_transform`、検証・テスト・本番データには `transform` のみ。


In [ ]:
from sklearn.preprocessing import StandardScaler

# OK: 学習データで fit、検証データには transform だけ
scaler = StandardScaler()
scaler.fit(df_imp.iloc[:600][['age', 'fare']])           # 学習データで基準を決める
X_tr_std = scaler.transform(df_imp.iloc[:600][['age', 'fare']])  # 学習データを変換
X_va_std = scaler.transform(df_imp.iloc[600:][['age', 'fare']])  # 検証データは同じ基準で変換

print('学習データに対する平均(ほぼ0):', X_tr_std.mean(axis=0).round(3))
print('検証データに対する平均(必ずしも0ではない):', X_va_std.mean(axis=0).round(3))


学習データから決めた基準値を使い回すので、検証データの平均は必ずしも0になりません。これが**正しい状態**です。もし検証データでも平均0になっていたら、それは検証データ側で `fit` をやり直してしまっている、つまりリーケージのサインです。


実務では `Pipeline` を使うと、CV の中で自動的に「fold ごとに train で fit、valid に transform」が行われます。**実務での標準パターン**として覚えておきましょう。


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# 補完・スケーリング・モデルをひとまとめにする
pre = ColumnTransformer([
    ('imputer', SimpleImputer(strategy='median'), ['age', 'fare']),
], remainder='passthrough')

pipe = Pipeline([
    ('pre', pre),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
])

# df(欠損のある元データ)で確認
X_raw = df[['age', 'fare', 'pclass', 'sibsp', 'parch']]
score_pipe = cross_val_score(pipe, X_raw, df['survived'], cv=5, scoring='accuracy').mean()
print(f'Pipeline + CV: accuracy = {score_pipe:.4f}')


このように、`Pipeline` を使えば fold ごとの `fit` / `transform` の使い分けを自動でやってくれるため、リーケージの心配なしに前処理と学習を回せます。


#### 練習問題 6

`random_id` のカテゴリ数を 50 / 200 / 500 と変えて、NG版のCVスコアがどう変わるか確認してみましょう。カテゴリ数が増えるほど1カテゴリあたりのサンプル数が減り、その分リーケージの影響、つまり見かけ上のスコア上昇が大きくなる様子を観察してください。

**ヒント**: `rng.randint(0, n, size=len(df_lk))` の `n` を変えてみてください。


In [ ]:
# 練習問題6の答えはこのセルに書いてください


# 7. 特徴量の選択と評価

本章では、ここまで作った特徴量を見直して、不要なものを減らす作業を実践してみましょう。相関ヒートマップで多重共線性を確認し、特徴量重要度と RFE で重要な特徴量を選んでいきます。最後に、章をまたいだCVスコアの累積効果を整理します。


まず、ここまで作った特徴量の相関ヒートマップを見て、似た情報を持つ特徴量があるか確認してみましょう。


In [ ]:
# Chapter 6 で y/X が一時的に上書きされたので、元に戻しておく
y = df['survived']

# 5章までの特徴量(X_ch5)で相関を見る
corr = X_ch5.corr().abs()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='Blues', annot=False, ax=ax, vmin=0, vmax=1)
ax.set_title('特徴量間の相関(絶対値)')
plt.show()


対角線以外で**色の濃い組み合わせ**、つまり相関の高い組み合わせがあれば、それらは似た情報を持つ特徴量です。たとえば `family_size` は `sibsp` と `parch` の和から作っているため、もとの列と相関が高くなりやすく、**片方を残すか検討する候補**になります。


次に Random Forest を学習して、どの特徴量が予測に効いているかを特徴量重要度で見てみましょう。


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200)
rf.fit(X_ch5, y)

importance = pd.Series(rf.feature_importances_, index=X_ch5.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('feature importance')
ax.set_title('特徴量重要度(Random Forest)')
plt.tight_layout()
plt.show()


棒の長い、つまり重要度の高い特徴量が予測に大きく寄与しています。逆に**棒の短い下位の特徴量は予測への寄与が小さく、削除候補**として検討できます。重要度はあくまで「見直しのヒント」であり、絶対的な正解ではない点に注意します。


次に RFE、つまり再帰的特徴消去で、重要度の低い特徴量を順に取り除き、10個に絞ってみましょう。


In [ ]:
from sklearn.feature_selection import RFE

rfe = RFE(estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100), n_features_to_select=10)
rfe.fit(X_ch5, y)

selected = X_ch5.columns[rfe.support_]
print('RFE で選ばれた特徴量(10個):')
for f in selected:
    print(f'  - {f}')


最後に、ここまでに作った特徴量がCVスコアに与えた累積効果を整理しましょう。ベースラインから章をまたいでスコアがどう変化したかが確認できます。


In [ ]:
experiments = {
    'ch1 ベースライン': X_baseline,
    'ch2 (+ 欠損処理)': X_ch2,
    'ch3 (+ 数値変換)': X_ch3,
    'ch4 (+ カテゴリエンコーディング)': X_ch4,
    'ch5 (+ 集約・仮説)': X_ch5,
    'ch7 RFE で10個に絞る': X_ch5[selected],
}

results = []
for label, X_ in experiments.items():
    for name, model in models.items():
        score = cross_val_score(model, X_, y, cv=5, scoring='accuracy').mean()
        results.append({'実験': label, 'モデル': name, 'CV accuracy': score})

results_df = pd.DataFrame(results).pivot(index='実験', columns='モデル', values='CV accuracy')
results_df.loc[list(experiments.keys())]  # 順序を保つ


章をまたいで特徴量を足していくとスコアが少しずつ伸び、RFEで10個に絞っても精度がほぼ維持される、あるいは向上することが見て取れます。**「特徴量は多ければ多いほど良い」のではなく、効くものを選んで残すことが大事**ということが、数値で確認できました。


### 補足:特徴量の評価はハイパーパラメータとセットで

本講義では比較を分かりやすくするため、モデルのハイパーパラメータを固定したまま各章のCVスコアを見てきました。しかし実務で「この特徴量を最終的に採用するか」を判断するときは、**特徴量を変えるたびにハイパーパラメータをチューニングし直し、それぞれの最良スコアどうしで比較する**のが基本です。固定のハイパーパラメータでは、特徴量が持つ本来の効果を引き出しきれず、誤って「効かない」と判断してしまうことがあります。


#### 練習問題 7

RFE で残す特徴量の数 `n_features_to_select` を 5, 10, 15, 20 と変えて、それぞれの場合のCVスコアを比較してみましょう。特徴量数を増やせば常に精度が上がるとは限らないことを確認してください。


In [ ]:
# 練習問題7の答えはこのセルに書いてください


謝辞：データセットの利用について

本教材では、データ分析・可視化の学習目的で、Seaborn の load_dataset 関数から取得できるサンプルデータセット titanic を使用しています。

取得元：seaborn-data
https://github.com/mwaskom/seaborn-data

seaborn-data は、Seaborn のサンプルコードやドキュメントで利用するためのデータ配布用リポジトリです。本教材では、出典を明記したうえで、教育目的の実習用データとして使用しています。
<br>